In [4]:
#= 
global variables ( variables which are accessable anywhere in the code) can only be read or print in a loop

To explicitely change the global variable we use the keywork "global" it tells julia that we are changing the global variable 
=#


count = 1 
while count <=10
    
    println(count)
    
    global count+= 1
end

1
2
3
4
5
6
7
8
9
10


In [8]:
 # IF-ELSE
name = "julia"
if name == "lulu" # notice that there is  no : ( unlike python)
    print("cutest of all time")
elseif name == "your mom"  # elseif comes if there are more than 2 conditions to check
    println("hello mom") 
elseif name =="julia"
    println("hello julia")
else 
    print("hehehehehe")
end # mandatory to write after every if else 

hello julia


In [13]:
A = [1 2 3;4 5 6;7 8 9]
S =size(A)
println(S[1])
println(S[2])


3
3
(3,)


In [102]:

function matmult( x:: Matrix{Float64} , y:: Matrix{Float64})
    
    x_dim =  size(x)
    y_dim =  size(y)
    if x_dim[2] != y_dim[1] 
        throw(ArgumentError("Invalid input detectet/dimensions not matching"))
    else 
        # xy = Matrix{Float64}(undef,x_dim[1],y_dim[2])
        xy = zeros(Float64,x_dim[1],y_dim[2])
        @inbounds @simd for i in 1:1:y_dim[2]  # i = column pointer
        @inbounds @simd    for j in 1:1:x_dim[2]  # j = pointer in row/column
             @inbounds @simd   for k in 1:1:x_dim[1] # k = row pointer
                    xy[k,i] += x[k,j]*y[j,i]
                
                end
            end
        end
        return xy
    end
end





    

matmult (generic function with 2 methods)

In [112]:
# 1. Create true Float64 matrices


function matmult_optimized(x::Matrix{Float64}, y::Matrix{Float64})
    x_dim = size(x)
    y_dim = size(y)
    
    if x_dim[2] != y_dim[1]
        throw(ArgumentError("Dimensions not matching"))
    end
    
    # 2. Use zeros so we can safely use += starting from 0.0
    # This creates exactly ONE allocation (~1.9 MiB)
    xy = zeros(Float64, x_dim[1], y_dim[2])
    
    for i in 1:y_dim[2]        # Outer loop
        for j in 1:x_dim[2]    # Middle loop
            # 3. Only apply macros to the innermost math loop!
            @inbounds @simd for k in 1:x_dim[1]
                xy[k, i] += x[k, j] * y[j, i]
            end
        end
    end
    return xy
end

# Test it now!


matmult_optimized (generic function with 1 method)

In [138]:
function add_(x:: Matrix{<:Number},y:: Matrix{<: Number})
    x_dim = size(x)
    y_dim = size(y)
    if x_dim != y_dim
        throw(ArgumentError("Dimension Mismatch"))
    else 
        xy = zeros(Float64,x_dim[1],x_dim[2])
        for i = 1:1:x_dim[2] # i is  column pointer
            for j = 1:1:x_dim[1] # j is row pointer
                xy[j,i] = x[j,i]+y[j,i]
            end
        end       
    end
    return xy
end

A = [1.0 2.0;
     3.0 4.0;
     5.0 6.0]
B = [1.0 2.0;
     3.0 4.0;
     5.0 6.0]
C = [1 2;
     3 4]
D = [1 2;
     3 4]
print(add_(A,B))
                

[2.0 4.0; 6.0 8.0; 10.0 12.0]

In [46]:
function transpose_(x::Matrix{<:Number})
    dims = size(x)
    xT = zeros(Float64, dims[2],dims[1])
    @inbounds @simd for i = 1:1:dims[1] 
       @inbounds @simd for j in 1:1:dims[2]  
                         xT[j,i] = x[i,j]
                              
        end
    end
return xT
end

transpose_ (generic function with 1 method)

In [42]:
v = [1,2,3,4,5]
u = [11.0,12.0,13.0,14.0,15.0]

function dot(u::Vector{<:Number},v::Vector{<:Number})
    ulen = size(u)[1]
    vlen = size(v)[1]
    if ulen != vlen
        throw(ArguementError("Dimension Mismatch"))
    else 
        a = 0
        for i in 1:1:ulen
            a += u[i]*v[i]
        end
    return a
    end
end
    
    
dot(u,v)        


205.0

In [45]:

#=A = [0 0 0;4 3 3;8 7 9]
sz = size(A)
function ref(A)
    
    for j in 1:1:sz[2]
        for i in 1:1:sz[1]
            if A[i,j] != 0 
                pivot = A[i,j]
                break
            else 
                if i == sz[1]
                    break 
                else 
                    A[i,j],A[i+1,j]=A[i+1,j],A[i,j]
                    j=1
                continue 
                end
            end
        end
    end
    return pivot
end
ref(A)

        
=#

9

In [96]:
A = [2 1 1;4 3 3;8 7 9]
A[1,:], A[2,:] =  A[2,:], A[1,:] 
println(A)
B =copy(A)
B[1,1,1] = 100
println(A)
println(B)

[4 3 3; 2 1 1; 8 7 9]
[4 3 3; 2 1 1; 8 7 9]
[100 3 3; 2 1 1; 8 7 9]


In [188]:
A = [2 1 1;4 3 3;8 7 9]
A_m = [1 2 3;4 5 6; 7 8 9]

function ref(u::Matrix{<:Number})
sz=size(A)
A_c = Float64.(u)
for i in 1:1:sz[1] # Selects/locks the row
    j_carry = 0
    pivot = 0
    for j in 1:1: sz[2] # surfs the locked row and finds the pivot
        if A_c[i,j] == 0            
            continue        
        else 
            pivot = A_c[i,j]
            j_carry = j
            break
        end
    end
    for k in i+1:1:sz[1]  # goes from the next row of the pivot row till the last row
        # multiplier = A[k,j]
        multiplier_eff = A_c[k,j_carry]/pivot
        for m in j_carry:1:sz[2] # performs elimination in the selected row ( from pivot row+1 till last row)
            A_c[k,m] = A_c[k,m] - multiplier_eff*A_c[i,m]
        end
    end
end
    return A_c
end
println(ref(A))
    
             
        
            
    


[2.0 1.0 1.0; 0.0 1.0 1.0; 0.0 0.0 2.0]


In [213]:
A = [2 1 1;4 3 3;8 7 9]
A_m = [1 2 3;4 5 6; 7 8 9]

function ref(u::Matrix{<:Number})
sz=size(u)
A_c = Float64.(u)
for i in 1:1:sz[1] # Selects/locks the row
    j_carry = 0
    pivot = 0
    for j in 1:1: sz[2] # surfs the locked row and finds the pivot
        if A_c[i,j] != 0            
            pivot = A_c[i,j]
            j_carry = j
            break       
        else 
            if i ==  sz[1]
                    break
            else
                    A_c[i,:], A_c[i+1,:] =  A_c[i+1,:], A_c[i,:] 
            end
            
        end
    end
    for k in i+1:1:sz[1]  # goes from the next row of the pivot row till the last row
        # multiplier = A[k,j]
        multiplier_eff = A_c[k,j_carry]/pivot
        for m in j_carry:1:sz[2] # performs elimination in the selected row ( from pivot row+1 till last row)
            A_c[k,m] = A_c[k,m] - multiplier_eff*A_c[i,m]
        end
    end
end
    return A_c
end
println(ref(A_m))
    
             
        
            
    


[1.0 2.0 3.0; 0.0 -6.0 -12.0; 0.0 0.0 0.0]


In [29]:
function ref(M :: Matrix{<:Number})
    U = Rational.(M)
    sz =size(U)
    i =1
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                                U[meow,:],U[i,:]=U[i,:],U[meow,:]
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return U
end

A_m = [3 6 5;0 1 7;7 8 9;1 7 2]
A = [2 1 1;4 3 3;8 7 9]

ref(A_m)

4×3 Matrix{Rational{Int64}}:
 3  6     5
 0  1     7
 0  0  118//3
 0  0     0

In [52]:
using BenchmarkTools

function ref(M :: Matrix{<:Number})
    U = Float64.(M)
    sz =size(U)
    i =1
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                               
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return U
end

A_m = [3 6 5;0 1 7;7 8 9;1 7 2]
A = [2 1 1;4 3 3;8 7 9]


M_large = rand(1:10,20,20)
@benchmark ref($M_large)

BenchmarkTools.Trial: 10000 samples with 5 evaluations per sample.
 Range (min … max):  5.920 μs … 768.100 μs  ┊ GC (min … max): 0.00% … 97.67%
 Time  (median):     6.340 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   8.604 μs ±  18.741 μs  ┊ GC (mean ± σ):  7.73% ±  3.61%

  ▇█▆▄▂▁▂▁▁   ▁▁▁▂▂▂▂▁▁   ▁▂▅▄▃▁                              ▂
  ██████████████████████████████▇▆▆▆▆▅▅▅▅▅▅▇▆▆▇▆▇▇▅▆▅▅▅▅▄▅▄▅▃ █
  5.92 μs      Histogram: log(frequency) by time      18.2 μs <

 Memory estimate: 5.40 KiB, allocs estimate: 23.

In [56]:
using BenchmarkTools
#= Best one so far 
speed : Float64>Int128,BigInt
GC impact : Int128>BigInt>Float64
Accuracy : BigInt>Int128> Float64
=#

function ref(M :: Matrix{<:Number})
    U = Rational{Int128}.(M)
    sz =size(U)
    i =1
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return U
end

A_m = [3 6 5;0 1 7;7 8 9;1 7 2]
A = [2 1 1;4 3 3;8 7 9]


M_large = rand(1:10,20,20)
@benchmark ref($M_large)

BenchmarkTools.Trial: 2507 samples with 1 evaluation per sample.
 Range (min … max):  1.739 ms …   3.077 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     2.043 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   1.993 ms ± 161.919 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

     ▂▄▂▂              ▃█▆▇▁                                   
  ▂▄█████▅▄▅▄▄▄▅▃▄▃▄▃▄▅█████▇▄▅▅▅▄▃▃▃▃▂▃▂▂▂▂▂▂▂▂▁▂▁▂▂▁▂▂▂▁▂▂▂ ▄
  1.74 ms         Histogram: frequency by time        2.57 ms <

 Memory estimate: 12.58 KiB, allocs estimate: 3.

In [50]:
using BenchmarkTools

function ref(M :: Matrix{<:Number})
    U = Rational{BigInt}.(M)
    sz =size(U)
    i =1
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                              
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return U
end

A_m = [3 6 5;0 1 7;7 8 9;1 7 2]
A = [2 1 1;4 3 3;8 7 9]


M_large = rand(1:10,20,20)
@benchmark ref($M_large)

BenchmarkTools.Trial: 560 samples with 1 evaluation per sample.
 Range (min … max):  3.273 ms … 320.349 ms  ┊ GC (min … max):  0.00% … 51.71%
 Time  (median):     5.874 ms               ┊ GC (median):     0.00%
 Time  (mean ± σ):   8.928 ms ±  28.152 ms  ┊ GC (mean ± σ):  16.99% ±  5.28%

          ▁       █▂                                           
  ▇▅▄▄▃▄▆▇█▆▇█▅▆▇▆██▇▆▆▅▆▆▆▆▅▇▇▆▅▆▄▄▄▃▃▂▃▃▁▂▂▁▁▂▁▁▂▁▂▁▁▁▁▂▁▁▂ ▃
  3.27 ms         Histogram: frequency by time        12.3 ms <

 Memory estimate: 2.24 MiB, allocs estimate: 71852.

In [88]:

function ref(M :: Matrix{<:Number})
    U = Rational{Int128}.(M)
    sz =size(U)
    i =1
    rank = 0
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    rank+=1
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end

    return U,rank,sz[2]-rank
end
A_m = [3 6 5;0 1 7;7 8 9;1 7 2]
A = [2 1 1;4 3 3;8 7 9]

U,r,n = ref(A)
println(U)
println("-----------------------------")
println(r)
println("-----------------------------")
println(n)



Rational{Int128}[2 1 1; 0 1 1; 0 0 2]
-----------------------------
3
-----------------------------
0


In [32]:

function ref(M :: Matrix{<:Number})
    U = Rational{Int128}.(M)
    sz =size(U)
    i =sz[1]
 while i>= 1 # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    break
                else
                    meow = i-1
                    
                    if meow <= 1
            while meow>= 1
                            if  U[meow,h] != 0
                            
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != 1
                                    meow-=1
                                else # meow == 1
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i-=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i-=1
    end
    return U
end

B = [3 6 5;0 1 7;7 8 9;1 7 2]
A = [2 1 1;4 3 3;8 7 9]
M = [1 1 7;5 12 7]

ref(M)[1]


1//1

In [34]:
U = [1 1 7;0 7 -28]


function rref(U::Matrix{<:Number})  
    D = Rational{Int128}.(U)
    sz = size(D)
    for i = sz[1]:-1:1 # goes from last row to the first row
        pivot = 0
        h_carry = 0
        for h in 1:1:sz[2] # traverses the selected row
            if D[i,h] != 0 # if a pivot is found then 
                h_carry = h
                break
            end
        end
        if h_carry != 0 # if h_carry is there then a pivot is found, hence only then we will proceed to do backward( OR upward) elimination 
            pivot = D[i,h_carry] 
            for meow in 1:1:sz[2]
                D[i,meow] = D[i,meow]/pivot
            end
            for j in i-1:-1:1 # selects/ locks the rows above the pivot row( selected by i pointer) 
                    mbp = D[j,h_carry]
                    for k in 1:1:sz[2]
                        D[j,k] = D[j,k]-mbp*D[i,k]
                    end
            end 
        end
    end
    return D
end
rref(U)
            
                
        

2×3 Matrix{Rational{Int128}}:
 1  0  11
 0  1  -4

In [89]:
using BenchmarkTools
function ref(M :: Matrix{<:Number})
    U = Rational{Int128}.(M)
    sz =size(U)
    i =1
    rank = 0
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    rank+=1
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot # mbp = multiplier by pivot i.e multiplier/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return (U,rank,sz[2]-rank)
end
#------------------------------------------------------------------------------------------------------------------------------

function rref(U::Matrix{<:Number})
     D_raw,rank,nullity = ref(U)
    D = Rational{Int128}.(D_raw)
    sz = size(D)
    for i = sz[1]:-1:1 # goes from last row to the first row
        pivot = 0
        h_carry = 0
        for h in 1:1:sz[2] # traverses the selected row
            if D[i,h] != 0 # if a pivot is found then 
                h_carry = h
                break
            end
        end
        if h_carry != 0 # if h_carry is there then a pivot is found, hence only then we will proceed to do backward( OR upward) elimination 
            pivot = D[i,h_carry] 
            for meow in 1:1:sz[2]
                D[i,meow] = D[i,meow]/pivot
            end
            for j in i-1:-1:1 # selects/ locks the rows above the pivot row( selected by i pointer) 
                    mbp = D[j,h_carry]
                    for k in 1:1:sz[2]
                        D[j,k] = D[j,k]-mbp*D[i,k]
                    end
            end 
        end
    end
    return D,rank,nullity
end

M = [1 2 3 0;4 5 6 0;7 8 9 0]


# 4x7 Augmented Matrix (6 coefficient columns + 1 constant column)
meow = [
    1  2  1  5  1  1;
    2  4  1  8  0  1;
    0  0  1  2  3  2;
    1  2 -1  1  1  3   
]

rref(meow)[1]


4×6 Matrix{Rational{Int128}}:
 1  2  0  3  0   1
 0  0  1  2  0  -1
 0  0  0  0  1   1
 0  0  0  0  0   0

In [64]:

function ref(M :: Matrix{<:Number})
    U = Rational{Int128}.(M)
    sz =size(U)
    i =1
    rank = 0
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    rank+=1
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot # mbp = multiplier by pivot i.e multiplier/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return (U,rank,sz[2]-rank)
end
#------------------------------------------------------------------------------------------------------------------------------

function rref(U::Matrix{<:Number})
     D_raw,rank,nullity = ref(U)
    D = Rational{Int128}.(D_raw)
    sz = size(D)
    for i = sz[1]:-1:1 # goes from last row to the first row
        pivot = 0
        h_carry = 0
        for h in 1:1:sz[2] # traverses the selected row
            if D[i,h] != 0 # if a pivot is found then 
                h_carry = h
                break
            end
        end
        if h_carry != 0 # if h_carry is there then a pivot is found, hence only then we will proceed to do backward( OR upward) elimination 
            pivot = D[i,h_carry] 
            for meow in 1:1:sz[2]
                D[i,meow] = D[i,meow]/pivot
            end
            for j in i-1:-1:1 # selects/ locks the rows above the pivot row( selected by i pointer) 
                    mbp = D[j,h_carry]
                    for k in 1:1:sz[2]
                        D[j,k] = D[j,k]-mbp*D[i,k]
                    end
            end 
        end
    end
    return D,rank,nullity
end




#--------------------------------------------------------------------------------------------------------------------------------------------

function solveLSE(D::Matrix{<:Number})
    M,rank,nullity = rref(D)
    sz = size(M)
    lr = @view M[sz[1],1:sz[2]-1]
    count = 0
    for i in lr 
        if i != 0
            count +=1
            break
        else
            continue
        end
    end
    if count ==0 && M[sz[1],sz[2]] != 0
        return "NO Solution"
    elseif rank < (sz[2]-1)
        return "infinite soln" , solutions(M)
    else
        
        return "Unique solution",solutions(M)
    end
end

M = [1 2 3 ;4 5 6 ;7 8 9]
# 4x7 Augmented Matrix (6 coefficient columns + 1 constant column)
Meow = [
    1  2  1  5  1  1   9;
    2  4  1  8  0  1  10;
    0  0  1  2  3  2  11;
    1  2 -1  1  1  3   5
]
rref(Meow)[1]

4×7 Matrix{Rational{Int128}}:
 1  2  0  3  0   1  4
 0  0  1  2  0  -1  2
 0  0  0  0  1   1  3
 0  0  0  0  0   0  0

In [65]:
Meow = [
    1  2  1  5  1  1   9;
    2  4  1  8  0  1  10;
    0  0  1  2  3  2  11;
    1  2 -1  1  1  3   5
]


    
M = Rational{Int128}.(Meow[:,1:size(Meow)[2]-1])   
# M = Rational{Int128}.(Meow[:, 1:size(Meow)[2]-1])




    
    

4×6 Matrix{Rational{Int128}}:
 1  2   1  5  1  1
 2  4   1  8  0  1
 0  0   1  2  3  2
 1  2  -1  1  1  3

In [42]:
# Pivot variables are x1, x3, x5 (columns 1, 3, 5 contain the leading 1s)
# Free variables are x2, x4, x6 (columns 2, 4, 6)
Meow = 
[
  1   2   0   3   0   1 9;  # Row 1 (Pivot x1 is here)
  0   0   1   2   0  -1 10;  # Row 2 (Pivot x3 is here)
  0   0   0   0   1   1 11;  # Row 3 (Pivot x5 is here)
  0   0   0   0   0   0  5  # Row 4 (Zero row)
]



function nullspace(D::Matrix{<:Number}) # D = rref(A)
    M_raw = D
    sz = size(M_raw)    
    M = Rational{Int128}.(M_raw[:,1:sz[2]-1])   
    column_status = zeros(Rational{Int128},sz[2]-1,2)
    for i in 1:1:sz[1]
        for j in 1:1:sz[2]-1
            if M[i,j] !=0
                column_status[j,1] = 1
                column_status[j,2]= i
                break
            end
        end
    end
   
           
                
    
    return  column_status    
  
end
nullspace(Meow)
    

6×2 Matrix{Rational{Int128}}:
 1  1
 0  0
 1  2
 0  0
 1  3
 0  0

In [38]:
# Pivot variables are x1, x3, x5 (columns 1, 3, 5 contain the leading 1s)
# Free variables are x2, x4, x6 (columns 2, 4, 6)
Meow = 
[
  1   2   0   3   0   1 9;  # Row 1 (Pivot x1 is here)
  0   0   1   2   0  -1 10;  # Row 2 (Pivot x3 is here)
  0   0   0   0   1   1 11;  # Row 3 (Pivot x5 is here)
  0   0   0   0   0   0  5  # Row 4 (Zero row)
]



function nullspace(D::Matrix{<:Number}) # D = rref(A)
    M_raw = D
    sz = size(M_raw)    
    M = Rational{Int128}.(M_raw[:,1:sz[2]-1])   
    column_status = zeros(Rational{Int128},sz[2]-1,2) # pivot column = 1 , free column = 0
    # first Column= keeps the check on pivot columns
    # second column  = keeps the track in which that pivot resides
    for i in 1:1:sz[1]
        for j in 1:1:sz[2]-1
            if M[i,j] !=0
                column_status[j,1] = 1
                column_status[j,2] =i
                break
            end
        end
    end
    linear_combination_of_vectors = Vector{Rational{Int128}}[]
    
    for j in 1:1:sz[2]-1
        basis_vector = zeros(Rational{Int128},sz[2]-1)
        if column_status[j,1] == 0
            basis_vector[j] = 1
        else
            row = column[j,2]
             
            







        
  return  linear_combination_of_vectors
end
nullspace(Meow)
    

LoadError: UndefVarError: `row_status` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [67]:
# Pivot variables are x1, x3, x5 (columns 1, 3, 5 contain the leading 1s)
# Free variables are x2, x4, x6 (columns 2, 4, 6)
Meow = 
[
  1   2   0   3   0   1 4;  # Row 1 (Pivot x1 is here)
  0   0   1   2   0  -1 2;  # Row 2 (Pivot x3 is here)
  0   0   0   0   1   1 3;  # Row 3 (Pivot x5 is here)
  0   0   0   0   0   0  0  # Row 4 (Zero row)
]


function nullspace(D::Matrix{<:Number}) # D = rref(A)
    M_raw = D
    sz = size(M_raw)    
    M = Rational{Int128}.(M_raw[:,1:sz[2]-1])   
    
    
    column_status = zeros(Int64, sz[2]-1, 2) # pivot column = 1 , free column = 0
    # first Column  = keeps the check on pivot columns
    # second column = keeps the track in which that pivot resides
    
    for i in 1:1:sz[1]
        for j in 1:1:sz[2]-1
            if M[i,j] != 0
                column_status[j,1] = 1
                column_status[j,2] = i
                break
            end
        end
    end #= By the end of this code block, our column_status Matrix(or array) should have the information of pivots in which column(in binary)
    and in which row =#
    
    linear_combination_of_vectors = Vector{Rational{Int128}}[]
    
    for j in 1:1:sz[2]-1
        if column_status[j,1] == 0 #= We only care about free columns where column 1 is 0 (i.e column_status[j,1] = 0)
                                      Because the pivot variabels of basis vector are (-1)*M[:,j] where j is a free column
                                    =#
            
            basis_vector = zeros(Rational{Int128}, sz[2]-1)
            basis_vector[j] = 1 # the variable of free column(free variable is set to 1 => this is linear algebra theory)
            
            # Now, look at all the pivot columns to fill in the rest of this vector
            for p_col in 1:1:sz[2]-1 # this loop traverses the columns 
                if column_status[p_col,1] == 1 
                    row = column_status[p_col, 2] #  note down the rows in which the pivot resides
                    basis_vector[p_col] = -M[row, j] # assign the element from the free column(after flipping the sign) to the pivot column element of the basis vector
                end
            end
            
            push!(linear_combination_of_vectors, basis_vector)
        end
    end
    
    
    
    return (particular_vector,"+", "linear combination of",linear_combination_of_vectors)
end
nullspace(Meow)
    

([4, 2, 3, 0], "+", "linear combination of", Vector{Rational{Int128}}[[-2, 1, 0, 0, 0, 0], [-3, 0, -2, 1, 0, 0], [-1, 0, 1, 0, -1, 1]])

In [83]:
function solutions(D::Matrix{<:Number}) # D = rref(A)
    M_raw = D
    sz = size(M_raw)    
    M = Rational{Int128}.(M_raw[:,1:sz[2]-1])   
    
    column_status = zeros(Int64, sz[2]-1, 2) # pivot column = 1 , free column = 0
    # first Column  = keeps the check on pivot columns
    # second column = keeps the track in which that pivot resides
    
    for i in 1:1:sz[1]
        for j in 1:1:sz[2]-1
            if M[i,j] != 0
                column_status[j,1] = 1
                column_status[j,2] = i
                break
            end
        end
    end #= By the end of this code block, our column_status Matrix(or array) should have the information of pivots in which column(in binary)
    and in which row =#

    
    particular_vector = zeros(Rational{Int128}, sz[2]-1) 
    for p_col in 1:1:sz[2]-1 # traverse the columns
        if column_status[p_col, 1] == 1     # if a pivot is found
            row = column_status[p_col, 2]   # then look for row in which that resides 
            particular_vector[p_col] = M_raw[row, sz[2]]  # then the particular vector will be the last element of that row

            # all free variables are set to be 0
        end
    end # particular vector is done after this code block

    
    linear_combination_of_vectors = Vector{Rational{Int128}}[]   
    for j in 1:1:sz[2]-1
        if column_status[j,1] == 0 #= We only care about free columns where column 1 is 0 (i.e column_status[j,1] = 0)
                                     Because the pivot variabels of basis vector are (-1)*M[:,j] where j is a free column
                                    =#
            
            basis_vector = zeros(Rational{Int128}, sz[2]-1)
            basis_vector[j] = 1 # the variable of free column(free variable is set to 1 => this is linear algebra theory)
            
            # Now, look at all the pivot columns to fill in the rest of this vector
            for p_col in 1:1:sz[2]-1 # this loop traverses the columns 
                if column_status[p_col,1] == 1 
                    row = column_status[p_col, 2] #  note down the rows in which the pivot resides
                    basis_vector[p_col] = -M[row, j] # assign the element from the free column(after flipping the sign) to the pivot column element of the basis vector
                end
            end
            
            push!(linear_combination_of_vectors, basis_vector)
        end
    end
    return (particular_vector , "+ linear combination of vectors",linear_combination_of_vectors)
end


Meow = 
[
  1   2   0   3   0   1 4;  # Row 1 (Pivot x1 is here)
  0   0   1   2   0  -1 2;  # Row 2 (Pivot x3 is here)
  0   0   0   0   1   1 3;  # Row 3 (Pivot x5 is here)
  0   0   0   0   0   0  0  # Row 4 (Zero row)
]

solutions(Meow)

(Rational{Int128}[4, 0, 2, 0, 3, 0], "+ linear combination of vectors", Vector{Rational{Int128}}[[-2, 1, 0, 0, 0, 0], [-3, 0, -2, 1, 0, 0], [-1, 0, 1, 0, -1, 1]])

In [86]:
using BenchmarkTools
function ref(M :: Matrix{<:Number})
    U = Float64.(M)
    sz =size(U)
    i =1
    rank = 0
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    rank+=1
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot # mbp = multiplier by pivot i.e multiplier/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return (U,rank,sz[2]-rank)
end
#------------------------------------------------------------------------------------------------------------------------------

function rref(U::Matrix{<:Number})
     D_raw,rank,nullity = ref(U)
    D = Float64.(D_raw)
    sz = size(D)
    for i = sz[1]:-1:1 # goes from last row to the first row
        pivot = 0
        h_carry = 0
        for h in 1:1:sz[2] # traverses the selected row
            if D[i,h] != 0 # if a pivot is found then 
                h_carry = h
                break
            end
        end
        if h_carry != 0 # if h_carry is there then a pivot is found, hence only then we will proceed to do backward( OR upward) elimination 
            pivot = D[i,h_carry] 
            for meow in 1:1:sz[2]
                D[i,meow] = D[i,meow]/pivot
            end
            for j in i-1:-1:1 # selects/ locks the rows above the pivot row( selected by i pointer) 
                    mbp = D[j,h_carry]
                    for k in 1:1:sz[2]
                        D[j,k] = D[j,k]-mbp*D[i,k]
                    end
            end 
        end
    end
    return D,rank,nullity
end




#--------------------------------------------------------------------------------------------------------------------------------------------

function solveLSE(D::Matrix{<:Number})
    M,rank,nullity = rref(D)
    sz = size(M)
    lr = @view M[sz[1],1:sz[2]-1]
    count = 0
    for i in lr 
        if i != 0
            count +=1
            break
        else
            continue
        end
    end
    if count ==0 && M[sz[1],sz[2]] != 0
        return ("NO Solution",)
    elseif rank < (sz[2]-1)
        return "infinite soln" , solutions(M,rank)
    else
        
        return "Unique solution",solutions(M,rank)
    end
end
#-------------------------------------------------------------------------------------------------------------------------------------------

function solutions(D::Matrix{<:Number},r::Number) # D = rref(A) and D is the augmented matrix i.e [A|b]
    M_raw = D
    rank =r 
    sz = size(M_raw)    
    M = Float64.(M_raw[:,1:sz[2]-1])   
    
    column_status = zeros(Int64, sz[2]-1, 2) # pivot column = 1 , free column = 0
    # first Column  = keeps the check on pivot columns
    # second column = keeps the track in which that pivot resides
    
    for i in 1:1:sz[1]
        for j in 1:1:sz[2]-1
            if M[i,j] != 0
                column_status[j,1] = 1
                column_status[j,2] = i
                break
            end
        end
    end #= By the end of this code block, our column_status Matrix(or array) should have the information of pivots in which column(in binary)
    and in which row =#

    
    particular_vector = zeros(Float64, sz[2]-1) 
    for p_col in 1:1:sz[2]-1 # traverse the columns
        if column_status[p_col, 1] == 1     # if a pivot is found
            row = column_status[p_col, 2]   # then look for row in which that resides 
            particular_vector[p_col] = M_raw[row, sz[2]]  # then the particular vector will be the last element of that row

            # all free variables are set to be 0
        end
    end # particular vector is done after this code block

    
    linear_combination_of_vectors = Vector{Float64}[]   
    for j in 1:1:sz[2]-1
        if column_status[j,1] == 0 #= We only care about free columns where column 1 is 0 (i.e column_status[j,1] = 0)
                                     Because the pivot variabels of basis vector are (-1)*M[:,j] where j is a free column
                                    =#
            
            basis_vector = zeros(Float64, sz[2]-1)
            basis_vector[j] = 1 # the variable of free column(free variable is set to 1 => this is linear algebra theory)
            
            # Now, look at all the pivot columns to fill in the rest of this vector
            for p_col in 1:1:sz[2]-1 # this loop traverses the columns 
                if column_status[p_col,1] == 1 
                    row = column_status[p_col, 2] #  note down the rows in which the pivot resides
                    basis_vector[p_col] = -M[row, j] # assign the element from the free column(after flipping the sign) to the pivot column element of the basis vector
                end
            end
            
            push!(linear_combination_of_vectors, basis_vector)
        end
    end
    if rank == sz[2]-1
        return particular_vector
    else
    return (particular_vector , "+ linear combination of vectors",linear_combination_of_vectors)
    end
end

Meow = [
    1  2  1  5  1  1   9;
    2  4  1  8  0  1  10;
    0  0  1  2  3  2  11;
    1  2 -1  1  1  3   5
]

C = [
    1  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  3;
0  1  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  5;
0  0  1  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  7;
0  0  0  1  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  9;
0  0  0  0  1  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  11;
0  0  0  0  0  1  1  0  0  0  0  0  0  0  0  0  0  0  0  0  13;
0  0  0  0  0  0  1  1  0  0  0  0  0  0  0  0  0  0  0  0  15;
0  0  0  0  0  0  0  1  1  0  0  0  0  0  0  0  0  0  0  0  17;
0  0  0  0  0  0  0  0  1  1  0  0  0  0  0  0  0  0  0  0  19;
0  0  0  0  0  0  0  0  0  1  1  0  0  0  0  0  0  0  0  0  21;
0  0  0  0  0  0  0  0  0  0  1  1  0  0  0  0  0  0  0  0  23;
0  0  0  0  0  0  0  0  0  0  0  1  1  0  0  0  0  0  0  0  25;
0  0  0  0  0  0  0  0  0  0  0  0  1  1  0  0  0  0  0  0  27;
0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  0  0  0  0  0  29;
0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  0  0  0  0  31;
0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  0  0  0  33;
0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  0  0  35;
0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  0  37;
0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  39;
1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  400;
]

JoJo =  [
    1.0  2.0  3.0;
    4.0  5.0  6.0
]


ref(JoJo)[1]




2×3 Matrix{Float64}:
 1.0   2.0   3.0
 0.0  -3.0  -6.0

In [6]:
using BenchmarkTools
Meow =[
     1.0   2.0   3.0;
 0.0  -3.0  -6.0;
 0.0   0.0   0.0
    ]

function det(M_raw ::Matrix{<:Number},r::Int64,n::Int64)
    M,rank, nullity = M_raw,r,n
    sz = size(M)

    if sz[2] != sz[1]
        return throw(ArgumentError("Non-Square Matrices dont have determinant"))
    elseif  rank<sz[2]
        return 0
    else
        determinant = 1
        for i in 1:sz[1]
            determinant *= M[i, i]
        end
    end
    return determinant
end

Meow_NonZero = [
    1.0  2.0  3.0;
    0.0 -3.0 -6.0;
    0.0  0.0  2.0
]

Meow_NotSquare = [1.0   2.0   3.0;
                  0.0  -3.0  -6.0
                                ]
@benchmark det($Meow_NonZero,3,0)

BenchmarkTools.Trial: 10000 samples with 1000 evaluations per sample.
 Range (min … max):  5.600 ns … 233.400 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     5.700 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   6.216 ns ±   4.042 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▅█▁                                                         ▁
  ███▆▁▅▅▄▅▁▄▆▆▇▁▇▇██▁████▁████▁████▁████▁██▇█▁███▆▁▆▆▆▆▁▆▅▆▄ █
  5.6 ns       Histogram: log(frequency) by time      10.3 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [11]:
function trace(M_raw ::Matrix{<:Number})
    M= M_raw
    sz = size(M)

    if sz[2] != sz[1]
        return throw(ArgumentError("Non-Square Matrices dont have Trace"))
    else
        trace = 0
        for i in 1:sz[1]
            trace += M[i, i]
        end
    end
    return trace
end

Meow_NonZero = [
    1.0  2.0  3.0;
    0.0 -4.0 -6.0;
    0.0  0.0  2.0
]
Meow_NotSquare = [1.0   2.0   3.0;
                  0.0  -3.0  -6.0
                                ]
trace(Meow_NonZero)

-1.0

In [1]:
using BenchmarkTools
function inverse(M::Matrix{<:Number})
    sz = size(M)
     U = Float64.(M)
    if sz[1] != sz[2]
        return throw(ArgumentError("Non Square Matrices dont have inverse"))
    #= elseif ref(M)[2] <sz[2]
        return "Inverse do NOT exist" =#
    else
       I_mat =  zeros(Float64,sz[1],sz[2])
        for i in 1:1:sz[1]
            I_mat[i,i] = 1
        end
          i =1
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp1 = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp1

                                temp2 = I_mat[meow, :]
                                I_mat[meow, :] = I_mat[i, :]
                                I_mat[i, :] = temp2
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end   
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot # mbp = multiplier by pivot i.e multiplier/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                     I_mat[j,k] = I_mat[j,k]-mbp*I_mat[i,k]
                end
            end
            i+=1
    end
    for i = sz[1]:-1:1 # goes from last row to the first row
        pivot = 0
        h_carry = 0
        for h in 1:1:sz[2] # traverses the selected row
            if U[i,h] != 0 # if a pivot is found then 
                h_carry = h
                break
            end
        end
        if h_carry != 0 # if h_carry is there then a pivot is found, hence only then we will proceed to do backward( OR upward) elimination 
            pivot = U[i,h_carry] 
            for meow in 1:1:sz[2]
                U[i,meow] = U[i,meow]/pivot
                I_mat[i,meow] = I_mat[i,meow]/pivot
            end
            for j in i-1:-1:1 # selects/ locks the rows above the pivot row( selected by i pointer) 
                    mbp = U[j,h_carry]
                    for k in 1:1:sz[2]
                        U[j,k] = U[j,k]-mbp*U[i,k]
                        I_mat[j,k] = I_mat[j,k]-mbp*I_mat[i,k]
                    end
            end 
        end
    end
    return I_mat
              
end  
end

A = [
    0.0 1.0;
    2.0 3.0
]
B = [
    0.0 0.0 1.0;
    0.0 2.0 3.0;
    4.0 5.0 6.0
]
@benchmark inverse($A)

BenchmarkTools.Trial: 10000 samples with 315 evaluations per sample.
 Range (min … max):  272.698 ns … 34.811 μs  ┊ GC (min … max):  0.00% … 98.52%
 Time  (median):     294.603 ns              ┊ GC (median):     0.00%
 Time  (mean ± σ):   419.235 ns ±  1.108 μs  ┊ GC (mean ± σ):  18.43% ±  7.40%

  ██▆▅▄▃▂▂▂▂▃▃▃▃▃▂▁                                            ▂
  ████████████████████▇█▆▇▇▅▆▆▅▆▅▅▆▅▅▆▅▆▆▅▆▃▅▅▅▃▄▄▄▄▁▄▃▃▅▃▁▄▃▄ █
  273 ns        Histogram: log(frequency) by time      1.24 μs <

 Memory estimate: 544 bytes, allocs estimate: 12.

In [25]:
I_mat =  zeros(Float64,3,3)
        for i in 1:1:3
            I_mat[i,i] = 1
        end
I_mat


3×3 Matrix{Float64}:
 1.0  0.0  0.0
 0.0  1.0  0.0
 0.0  0.0  1.0

In [ ]:
  U = Float64.(M)
    sz =size(U)
    i =1
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot # mbp = multiplier by pivot i.e multiplier/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return U
end
#----------------------------------------------------------------------------------------------------------------------------------------------

 D_raw,rank,nullity = ref(U)
    D = Float64.(D_raw)
    sz = size(D)
    for i = sz[1]:-1:1 # goes from last row to the first row
        pivot = 0
        h_carry = 0
        for h in 1:1:sz[2] # traverses the selected row
            if D[i,h] != 0 # if a pivot is found then 
                h_carry = h
                break
            end
        end
        if h_carry != 0 # if h_carry is there then a pivot is found, hence only then we will proceed to do backward( OR upward) elimination 
            pivot = D[i,h_carry] 
            for meow in 1:1:sz[2]
                D[i,meow] = D[i,meow]/pivot
            end
            for j in i-1:-1:1 # selects/ locks the rows above the pivot row( selected by i pointer) 
                    mbp = D[j,h_carry]
                    for k in 1:1:sz[2]
                        D[j,k] = D[j,k]-mbp*D[i,k]
                    end
            end 
        end
    end
    return D,rank,nullity
end